In [ ]:
### this is to just login into briker terminal . make sure whenever we login for first time 
# place an dummy order get Super token by clicking shit+ ctrl+i --> network --> payload .


import csv
from datetime import datetime as dt_datetime, timedelta
import time
import threading
import logging
import pandas as pd
from NorenRestApiPy.NorenApi import NorenApi
import pyotp
import yaml
import token
from dateutil.relativedelta import relativedelta


class ShoonyaApiPy(NorenApi):
    def __init__(self):
        # super().__init__(host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')
        super().__init__(host='https://trade.shoonya.com/NorenWClientWeb/', websocket='wss://trade.shoonya.com/NorenWSWeb/')



# Initialize API
api = ShoonyaApiPy()

with open('cred.yml') as f:
    cred = yaml.load(f, Loader=yaml.FullLoader)

SuperToken ="31c954501666f2a546ef4384b9bc79c2f6273440ca3065b63da6f8d14d01bebd"
userId=cred['user']
Passwrd=cred['pwd']
ret = api.set_session(userId , Passwrd ,SuperToken )


if ret:
    print("Login Successful")
else:
    print("Login Failed")
    exit()

print(ret)



In [ ]:
################################################################################
#################################################################################
# 1

# 1st script to run to get the stocks_info list which will be used in backtesting. 
# You can copy the output list and paste it in the backtesting code. Make sure to have the filtered_stocks.csv file ready with the correct data for this to work.

################################################################################
#################################################################################





import pandas as pd
import os
from io import StringIO

# print("=" * 80)
# print("FULL PIPELINE: FILTER + LIMIT + EXTRACT stocks_info")
# print("=" * 80)
# print()

# 🔹 FILE PATHS
input_csv = r'C:\Users\omkar\Downloads\Backtest bb_blast_sell_Combined.csv'
filtered_csv = r'C:\Users\omkar\Downloads\filtered_stocks.csv'

# ==================== STEP 1: FILTER BACKTEST CSV ====================
def process_and_save(file_path, save_path):
    try:
        df = pd.read_csv(file_path)

        # print("\n📥 Raw Data:")
        # print(df.head())

        # ✅ Convert to datetime (your format)
        df.iloc[:, 0] = pd.to_datetime(
            df.iloc[:, 0],
            format='%d-%m-%Y %I:%M %p',
            errors='coerce'
        )

        # Force dtype
        df[df.columns[0]] = pd.to_datetime(df[df.columns[0]])

        # Remove invalid rows
        df = df.dropna(subset=[df.columns[0]])

        # Extract time
        df['time_only'] = df.iloc[:, 0].dt.strftime('%H:%M')

        # print("\n🕒 Time distribution:")
        # print(df['time_only'].value_counts())

        # Step 1: Filter required times
        valid_times = ['09:45', '10:00', '10:30', '10:45']
        filtered_df = df[df['time_only'].isin(valid_times)].copy()

        # print("\n✅ After Time Filter:")
        # print(filtered_df)

        # ==================== NEW RULE ====================
        # Keep only timestamps where stocks <= 3
        filtered_df['timestamp_str'] = filtered_df.iloc[:, 0].astype(str)

        counts = filtered_df['timestamp_str'].value_counts()

        # print("\n📊 Stocks per timestamp:")
        # print(counts)

        # Keep only timestamps with <= 3 stocks
        valid_timestamps = counts[counts <= 3].index

        filtered_df = filtered_df[filtered_df['timestamp_str'].isin(valid_timestamps)]

        # print("\n✅ After Applying Max 3 Stocks Rule:")
        # print(filtered_df)

        # Save filtered CSV
        filtered_df.to_csv(save_path, index=False)
        # print(f"\n💾 Saved filtered file: {save_path}")

        return filtered_df

    except Exception as e:
        print(f"\n❌ ERROR in filtering: {e}")
        return None


# ==================== STEP 2: EXTRACTION ====================
def extract_from_csv_file(file_path):
    try:
        df = pd.read_csv(file_path)

        stocks_info = []
        for _, row in df.iterrows():
            stocks_info.append((str(row.iloc[0]).strip(), str(row.iloc[1]).strip()))

        return stocks_info

    except Exception as e:
        print(f"✗ Error reading CSV: {e}")
        return []


# ==================== STEP 3: RUN FILTER ====================
filtered_df = process_and_save(input_csv, filtered_csv)

# ==================== STEP 4: LOAD FILTERED CSV ====================
stocks_info = []

if os.path.exists(filtered_csv):
    print("\n📁 Loading from filtered CSV...")

    stocks_info = extract_from_csv_file(filtered_csv)

    if stocks_info:
        print(f"✓ Loaded {len(stocks_info)} stocks")
    else:
        print("❌ No data found in filtered CSV")
else:
    print("❌ Filtered CSV not found")

# ==================== FINAL OUTPUT ====================
print("\n" + "=" * 80)
print("📦 FINAL stocks_info")
print("=" * 80)

if stocks_info:
    print("stocks_info = [")
    for dt, sym in stocks_info:
        print(f"    ('{dt}', '{sym}'),")
    print("]")
else:
    print("❌ No stocks found")

print("=" * 80)
print(f"📊 Total stocks: {len(stocks_info)}")
print("=" * 80)

In [ ]:

######################################################################################
#2 
        # This script is used for downloading historical data for a list of stocks using 
        # the Shoonya API.
        #it need Stocks_info which needs to be updated from other script 

#######################################################################################
    


import csv
from datetime import datetime, timedelta
import time
import threading
import logging
import pandas as pd
from NorenRestApiPy.NorenApi import NorenApi
import pyotp
import yaml
import token
from dateutil.relativedelta import relativedelta


print("=" * 80)
print("COMPLETE WORKFLOW: LOGIN → EXTRACT SYMBOLS → DOWNLOAD DATA")
print("=" * 80)
print()

# ==================== STEP 1: EXTRACT SYMBOLS WITH -EQ SUFFIX ====================
print("📊 STEP 1: EXTRACTING SYMBOLS FROM stocks_info")
print("=" * 80)

symbols_array = []
stocks_info = [
    ('2026-01-07 10:45:00', 'SUNPHARMA'),
    ('2026-01-07 10:45:00', 'NATCOPHARM'),
    ('2026-01-08 10:00:00', 'GLAND'),
    ('2026-01-14 10:00:00', 'BLS'),
    ('2026-01-16 10:00:00', 'TECHM'),
    ('2026-01-16 10:45:00', 'AMBUJACEM'),
    ('2026-01-19 10:00:00', 'NETWEB'),
    ('2026-01-22 10:30:00', 'ECLERX'),
    ('2026-01-23 10:00:00', 'HINDCOPPER'),
    ('2026-01-23 10:00:00', 'HOMEFIRST'),
    ('2026-01-23 10:30:00', 'SYRMA'),
    ('2026-01-28 09:45:00', 'NBCC'),
    ('2026-01-28 10:00:00', 'VEDL'),
    ('2026-01-28 10:00:00', 'SAMMAANCAP'),
    ('2026-01-28 10:00:00', 'IRCON'),
    ('2026-01-29 09:45:00', 'ACUTAAS'),
    ('2026-01-30 10:45:00', 'TECHNOE'),
    ('2026-02-01 09:45:00', 'DOMS'),
    ('2026-02-01 10:00:00', 'HBLENGINE'),
    ('2026-02-03 10:30:00', 'CUB'),
    ('2026-02-03 10:30:00', 'TARIL'),
    ('2026-02-03 10:30:00', 'TRITURBINE'),
    ('2026-02-04 10:00:00', 'APOLLOTYRE'),
    ('2026-02-04 10:00:00', 'GVT&D'),
    ('2026-02-04 10:30:00', 'NCC'),
    ('2026-02-04 10:30:00', 'RECLTD'),
    ('2026-02-05 09:45:00', 'BPCL'),
    ('2026-02-05 09:45:00', 'CCL'),
    ('2026-02-05 10:00:00', 'CHENNPETRO'),
    ('2026-02-05 10:00:00', 'DEVYANI'),
    ('2026-02-09 09:45:00', 'IRFC'),
    ('2026-02-12 10:00:00', 'ELGIEQUIP'),
    ('2026-02-12 10:00:00', 'ENRIN'),
    ('2026-02-17 10:00:00', 'BEL'),
    ('2026-02-17 10:00:00', 'KIRLOSENG'),
    ('2026-02-18 10:30:00', 'ITC'),
    ('2026-02-19 09:45:00', 'HFCL'),
    ('2026-02-19 10:00:00', 'LICHSGFIN'),
    ('2026-02-24 10:45:00', 'J&KBANK'),
    ('2026-02-25 09:45:00', 'KAYNES'),
    ('2026-02-26 10:30:00', 'GLENMARK'),
    ('2026-03-05 10:00:00', 'TATAINVEST'),
    ('2026-03-06 10:00:00', 'TATAELXSI'),
    ('2026-03-06 10:00:00', 'LTFOODS'),
    ('2026-03-06 10:00:00', 'PFC'),
    ('2026-03-10 09:45:00', 'NTPCGREEN'),
    ('2026-03-10 10:30:00', 'TEGA'),
    ('2026-03-11 09:45:00', 'TECHM'),
    ('2026-03-11 09:45:00', 'LTM'),
    ('2026-03-11 10:30:00', 'SCI'),
    ('2026-03-12 10:00:00', 'JINDALSAW'),
    ('2026-03-13 09:45:00', 'POWERGRID'),
    ('2026-03-17 10:00:00', 'PVRINOX'),
    ('2026-03-17 10:30:00', 'JYOTICNC'),
    ('2026-03-20 10:30:00', 'FIRSTCRY'),
    ('2026-03-24 09:45:00', 'CANFINHOME'),
    ('2026-03-24 10:00:00', 'UNOMINDA'),
    ('2026-03-24 10:00:00', 'TIINDIA'),
    ('2026-03-24 10:00:00', 'TENNIND'),
    ('2026-03-24 10:45:00', 'BLS'),
    ('2026-03-25 10:00:00', 'POLICYBZR'),
    ('2026-03-25 10:30:00', 'ABREL'),
    ('2026-03-25 10:30:00', 'GABRIEL'),
    ('2026-03-30 10:00:00', 'CHENNPETRO'),
    ('2026-04-01 09:45:00', 'ASTERDM'),
    ('2026-04-01 09:45:00', 'THELEELA'),
    ('2026-04-01 10:30:00', 'BDL'),
    ('2026-04-08 09:45:00', 'EMMVEE'),
    ('2026-04-09 10:30:00', 'MEESHO'),
    ('2026-04-10 09:45:00', 'ADANIPOWER'),
    ('2026-04-15 10:30:00', 'PWL'),
    ('2026-04-16 10:45:00', 'GICRE'),
    ('2026-04-17 09:45:00', 'CESC'),
    ('2026-04-17 09:45:00', 'NCC'),
    ('2026-04-17 10:30:00', 'FIRSTCRY'),
    ('2026-04-17 10:45:00', 'IRFC'),
    ('2026-04-20 10:30:00', 'AIAENG'),
    ('2026-04-21 10:00:00', 'BSOFT'),
    ('2026-04-21 10:00:00', 'PRESTIGE'),
    ('2026-04-21 10:00:00', 'CPPLUS'),
    ('2026-04-21 10:30:00', 'SOBHA'),
    ('2026-04-21 10:45:00', 'RKFORGE'),
    ('2026-04-22 10:00:00', 'TATAINVEST'),
    ('2026-04-22 10:00:00', 'MANAPPURAM'),
    ('2026-04-22 10:00:00', 'GRSE'),
    ('2026-04-22 10:45:00', 'RHIM'),
    ('2026-04-23 10:00:00', 'EIDPARRY'),
    ('2026-04-23 10:30:00', 'GVT&D'),
    ('2026-04-23 10:45:00', 'MANKIND'),
    ('2026-04-24 10:00:00', 'HSCL'),
    ('2026-04-24 10:00:00', 'COALINDIA'),
    ('2026-04-24 10:00:00', 'IKS'),
    ('2026-04-27 10:00:00', 'LLOYDSME'),
    ('2026-04-27 10:00:00', 'PNBHOUSING'),
    ('2026-04-27 10:30:00', 'POLICYBZR'),
    ('2026-04-29 10:00:00', 'LEMONTREE'),
    ('2026-04-29 10:00:00', 'TENNIND'),
    ('2026-04-29 10:30:00', 'SONATSOFTW'),
    ('2026-04-30 09:45:00', 'TATACHEM'),
    ('2026-04-30 10:00:00', 'BAJFINANCE'),
    ('2026-04-30 10:00:00', 'SYNGENE'),
    ('2026-04-30 10:00:00', 'CREDITACC'),
]
for _, stock_symbol in stocks_info:
    stock_with_suffix = f"{stock_symbol}-EQ"
    if stock_with_suffix not in symbols_array:  # Avoid duplicates
        symbols_array.append(stock_with_suffix)

symbols_array.sort()  # Sort alphabetically

print(f"✓ Extracted {len(symbols_array)} unique stocks with -EQ suffix")
print()
print("Symbols to download:")
for i, symbol in enumerate(symbols_array, 1):
    print(f"  {i}. {symbol}")
print()

# ==================== STEP 2: BROKER API LOGIN ====================
print("=" * 80)
print("🔐 STEP 2: INITIALIZING BROKER API")
print("=" * 80)
print()




# ==================== STEP 3: PREPARE DOWNLOAD PARAMETERS ====================
print("=" * 80)
print("📅 STEP 3: PREPARING DATA DOWNLOAD PARAMETERS")
print("=" * 80)

# Calculate date range (4 months back)
now = datetime.now()
start_date = now - relativedelta(months=4)
start_date = start_date.replace(hour=0, minute=0, second=0, microsecond=0)
start_timestamp = start_date.timestamp()

print(f"Start date: {datetime.fromtimestamp(start_timestamp)}")
print(f"End date: {now}")
print(f"Data period: 4 months")
print(f"Exchange: NSE")
print(f"Interval: 1 minute")
print()

# ==================== STEP 4: DATA OUTPUT DIRECTORY ====================
output_directory = r'D:\AlgoRepo\ShoonyaAPI_Code\Testing_Use\Stocks_DATA'
print(f"Output directory: {output_directory}")
print()

# Create directory if it doesn't exist
import os
os.makedirs(output_directory, exist_ok=True)
print(f"✓ Directory ready")
print()

# ==================== STEP 5: DOWNLOAD HISTORICAL DATA ====================
print("=" * 80)
print("📥 STEP 5: DOWNLOADING HISTORICAL DATA")
print("=" * 80)
print()

print("Clearing old files...")
deleted_count = 0
failed_count = 0
for file in os.listdir(output_directory):
    file_path = os.path.join(output_directory, file)
    if os.path.isfile(file_path):
        try:
            os.remove(file_path)
            deleted_count += 1
        except PermissionError as e:
            print(f"⚠️  Could not delete {file} (file is locked). Skipping...")
            failed_count += 1
            
if deleted_count > 0:
    print(f"✓ Deleted {deleted_count} old files")
if failed_count > 0:
    print(f"⚠️  {failed_count} files could not be deleted (locked/in use)")
print()

successful_downloads = []
failed_downloads = []

for idx, stock in enumerate(symbols_array, 1):
    try:
        print(f"[{idx}/{len(symbols_array)}] Downloading {stock}...", end=" ")
        
        # Fetch the data from the API
        ret = api.get_time_price_series(exchange='NSE', token=stock, starttime=start_timestamp, interval=1)
        
        if ret and len(ret) > 0:
            # Convert the response into a DataFrame
            df_stock = pd.DataFrame(ret)
            
            # Save the DataFrame to an Excel file
            output_file_path = f'{output_directory}\\{stock}.xlsx'
            df_stock.to_excel(output_file_path, index=False)
            
            print(f"✓ ({len(df_stock)} rows saved)")
            successful_downloads.append(stock)
            
        else:
            print(f"✗ (No data returned)")
            failed_downloads.append(stock)
            
    except Exception as e:
        print(f"✗ (Error: {str(e)[:50]})")
        failed_downloads.append(stock)
        logging.error(f"Error downloading {stock}: {e}")

print()

# ==================== STEP 6: DOWNLOAD SUMMARY ====================
print("=" * 80)
print("✅ DOWNLOAD COMPLETE - SUMMARY REPORT")
print("=" * 80)
print()

print(f"📊 STATISTICS:")
print(f"   Total stocks: {len(symbols_array)}")
print(f"   Successfully downloaded: {len(successful_downloads)}")
print(f"   Failed downloads: {len(failed_downloads)}")
success_rate = (len(successful_downloads)/len(symbols_array)*100) if len(symbols_array) > 0 else 0
print(f"   Success rate: {success_rate:.1f}%")
print()

if successful_downloads:
    print(f"✓ SUCCESSFUL DOWNLOADS ({len(successful_downloads)}):")
    for i, stock in enumerate(successful_downloads, 1):
        print(f"   {i}. {stock}")
    print()

if failed_downloads:
    print(f"✗ FAILED DOWNLOADS ({len(failed_downloads)}):")
    for i, stock in enumerate(failed_downloads, 1):
        print(f"   {i}. {stock}")
    print()

print("=" * 80)
print(f"📁 Files saved to: {output_directory}")
print(f"🎯 All data ready for backtesting!")
print("=" * 80)



In [ ]:
###############################################################################

#3

#this is code whre you need to paste in ('13-09-2024 10:00', 'MANAPPURAM'), this way and make sure all 
# the stocks are placed in folder whih will give the output with backtest .

################################################################################ 



import pandas as pd
import os

# Set the directory where the Excel sheets are stored
excel_directory = 'D:\\AlgoRepo\\ShoonyaAPI_Code\\Testing_Use\\Stocks_DATA'

# Stock info in format: (entry time, stock symbol)
stocks_info = [
    ('2026-01-07 10:45:00', 'SUNPHARMA'),
    ('2026-01-07 10:45:00', 'NATCOPHARM'),
    ('2026-01-08 10:00:00', 'GLAND'),
    ('2026-01-14 10:00:00', 'BLS'),
    ('2026-01-16 10:00:00', 'TECHM'),
    ('2026-01-16 10:45:00', 'AMBUJACEM'),
    ('2026-01-19 10:00:00', 'NETWEB'),
    ('2026-01-22 10:30:00', 'ECLERX'),
    ('2026-01-23 10:00:00', 'HINDCOPPER'),
    ('2026-01-23 10:00:00', 'HOMEFIRST'),
    ('2026-01-23 10:30:00', 'SYRMA'),
    ('2026-01-28 09:45:00', 'NBCC'),
    ('2026-01-28 10:00:00', 'VEDL'),
    ('2026-01-28 10:00:00', 'SAMMAANCAP'),
    ('2026-01-28 10:00:00', 'IRCON'),
    ('2026-01-29 09:45:00', 'ACUTAAS'),
    ('2026-01-30 10:45:00', 'TECHNOE'),
    ('2026-02-01 09:45:00', 'DOMS'),
    ('2026-02-01 10:00:00', 'HBLENGINE'),
    ('2026-02-03 10:30:00', 'CUB'),
    ('2026-02-03 10:30:00', 'TARIL'),
    ('2026-02-03 10:30:00', 'TRITURBINE'),
    ('2026-02-04 10:00:00', 'APOLLOTYRE'),
    ('2026-02-04 10:00:00', 'GVT&D'),
    ('2026-02-04 10:30:00', 'NCC'),
    ('2026-02-04 10:30:00', 'RECLTD'),
    ('2026-02-05 09:45:00', 'BPCL'),
    ('2026-02-05 09:45:00', 'CCL'),
    ('2026-02-05 10:00:00', 'CHENNPETRO'),
    ('2026-02-05 10:00:00', 'DEVYANI'),
    ('2026-02-09 09:45:00', 'IRFC'),
    ('2026-02-12 10:00:00', 'ELGIEQUIP'),
    ('2026-02-12 10:00:00', 'ENRIN'),
    ('2026-02-17 10:00:00', 'BEL'),
    ('2026-02-17 10:00:00', 'KIRLOSENG'),
    ('2026-02-18 10:30:00', 'ITC'),
    ('2026-02-19 09:45:00', 'HFCL'),
    ('2026-02-19 10:00:00', 'LICHSGFIN'),
    ('2026-02-24 10:45:00', 'J&KBANK'),
    ('2026-02-25 09:45:00', 'KAYNES'),
    ('2026-02-26 10:30:00', 'GLENMARK'),
    ('2026-03-05 10:00:00', 'TATAINVEST'),
    ('2026-03-06 10:00:00', 'TATAELXSI'),
    ('2026-03-06 10:00:00', 'LTFOODS'),
    ('2026-03-06 10:00:00', 'PFC'),
    ('2026-03-10 09:45:00', 'NTPCGREEN'),
    ('2026-03-10 10:30:00', 'TEGA'),
    ('2026-03-11 09:45:00', 'TECHM'),
    ('2026-03-11 09:45:00', 'LTM'),
    ('2026-03-11 10:30:00', 'SCI'),
    ('2026-03-12 10:00:00', 'JINDALSAW'),
    ('2026-03-13 09:45:00', 'POWERGRID'),
    ('2026-03-17 10:00:00', 'PVRINOX'),
    ('2026-03-17 10:30:00', 'JYOTICNC'),
    ('2026-03-20 10:30:00', 'FIRSTCRY'),
    ('2026-03-24 09:45:00', 'CANFINHOME'),
    ('2026-03-24 10:00:00', 'UNOMINDA'),
    ('2026-03-24 10:00:00', 'TIINDIA'),
    ('2026-03-24 10:00:00', 'TENNIND'),
    ('2026-03-24 10:45:00', 'BLS'),
    ('2026-03-25 10:00:00', 'POLICYBZR'),
    ('2026-03-25 10:30:00', 'ABREL'),
    ('2026-03-25 10:30:00', 'GABRIEL'),
    ('2026-03-30 10:00:00', 'CHENNPETRO'),
    ('2026-04-01 09:45:00', 'ASTERDM'),
    ('2026-04-01 09:45:00', 'THELEELA'),
    ('2026-04-01 10:30:00', 'BDL'),
    ('2026-04-08 09:45:00', 'EMMVEE'),
    ('2026-04-09 10:30:00', 'MEESHO'),
    ('2026-04-10 09:45:00', 'ADANIPOWER'),
    ('2026-04-15 10:30:00', 'PWL'),
    ('2026-04-16 10:45:00', 'GICRE'),
    ('2026-04-17 09:45:00', 'CESC'),
    ('2026-04-17 09:45:00', 'NCC'),
    ('2026-04-17 10:30:00', 'FIRSTCRY'),
    ('2026-04-17 10:45:00', 'IRFC'),
    ('2026-04-20 10:30:00', 'AIAENG'),
    ('2026-04-21 10:00:00', 'BSOFT'),
    ('2026-04-21 10:00:00', 'PRESTIGE'),
    ('2026-04-21 10:00:00', 'CPPLUS'),
    ('2026-04-21 10:30:00', 'SOBHA'),
    ('2026-04-21 10:45:00', 'RKFORGE'),
    ('2026-04-22 10:00:00', 'TATAINVEST'),
    ('2026-04-22 10:00:00', 'MANAPPURAM'),
    ('2026-04-22 10:00:00', 'GRSE'),
    ('2026-04-22 10:45:00', 'RHIM'),
    ('2026-04-23 10:00:00', 'EIDPARRY'),
    ('2026-04-23 10:30:00', 'GVT&D'),
    ('2026-04-23 10:45:00', 'MANKIND'),
    ('2026-04-24 10:00:00', 'HSCL'),
    ('2026-04-24 10:00:00', 'COALINDIA'),
    ('2026-04-24 10:00:00', 'IKS'),
    ('2026-04-27 10:00:00', 'LLOYDSME'),
    ('2026-04-27 10:00:00', 'PNBHOUSING'),
    ('2026-04-27 10:30:00', 'POLICYBZR'),
    ('2026-04-29 10:00:00', 'LEMONTREE'),
    ('2026-04-29 10:00:00', 'TENNIND'),
    ('2026-04-29 10:30:00', 'SONATSOFTW'),
    ('2026-04-30 09:45:00', 'TATACHEM'),
    ('2026-04-30 10:00:00', 'BAJFINANCE'),
    ('2026-04-30 10:00:00', 'SYNGENE'),
    ('2026-04-30 10:00:00', 'CREDITACC'),
]

# Result storage
results = []

# Initial investment per stock
initial_investment = 200000

# Define 15:15 time for cutting positions
cutoff_time = pd.to_datetime('2026-03-25 15:15', format='%Y-%m-%d %H:%M')

# Iterate over each stock and its respective entry time
for entry_time_str, stock_name in stocks_info:

    # Files are named with -EQ suffix (e.g., MRPL-EQ.xlsx)
    excel_file_name = f"{stock_name}-EQ.xlsx"
    excel_file_path = os.path.join(excel_directory, excel_file_name)

    try:
        # Load stock data
        stock_data = pd.read_excel(excel_file_path, usecols=['time', 'intc'])
        stock_data['time'] = pd.to_datetime(stock_data['time'], format='%d-%m-%Y %H:%M:%S', errors='coerce')

        # Sort the data, ensuring latest data is processed correctly
        stock_data.sort_values(by='time', ascending=True, inplace=True)

        # Convert entry_time_str to timestamp
        entry_time = pd.to_datetime(entry_time_str, format='%Y-%m-%d %H:%M:%S')

        # Add 2 minutes to the entry time for testing
        testing_start_time = entry_time + pd.Timedelta(minutes=2)

        # Use .iloc[0] to get first row after testing_start_time
        entry_rows = stock_data[stock_data['time'] >= testing_start_time]
        
        if not entry_rows.empty:
            entry_row = entry_rows.iloc[0]
            entry_price = entry_row['intc']
            qty = int(initial_investment / entry_price)  # Calculate quantity of stocks

            # # For SHORT SELLING
            # profit_target = entry_price * 0.988  # Target at 0.7% LOWER (profit on short)
            # stop_loss = entry_price * 1.006    # Stop loss at 1.5% HIGHER (protect from rise)
            # For LONG SELLING
            profit_target = entry_price * 0.988 #get at 0.7% LOWER (profit on short)
            stop_loss = entry_price * 1.005 # Stop loss at 1.5% HIGHER (ssprotect from rise)

            # Filter subsequent data (after the entry row)
            subsequent_data = stock_data[stock_data['time'] > entry_row['time']]

            stop_or_tgt_hit = False
            for index, row in subsequent_data.iterrows():
                current_price = row['intc']
                current_time = row['time']

                # Check profit target first (price BELOW entry = profit on short)
                if current_price < profit_target:
                    profit_loss_amount = ((entry_price - current_price) * qty)
                    results.append([stock_name, entry_time, current_time, current_price, 'Profit', profit_loss_amount])
                    print(f"Profit target hit at {current_time}: {current_price:.2f}, Profit: {profit_loss_amount:.2f}")
                    stop_or_tgt_hit = True
                    break

                # Check stop loss (price ABOVE entry = loss on short)
                elif current_price > stop_loss:
                    profit_loss_amount = ((entry_price - current_price) * qty)
                    results.append([stock_name, entry_time, current_time, current_price, 'Loss', profit_loss_amount])
                    print(f"Stop loss hit at {current_time}: {current_price:.2f}, Loss: {profit_loss_amount:.2f}")
                    stop_or_tgt_hit = True
                    break

            # Ensure the trade is closed at cutoff time if no stop-loss or target is hit
            cutoff_rows = stock_data[stock_data['time'] >= cutoff_time]
            if not stop_or_tgt_hit and not cutoff_rows.empty:
                cutoff_price = cutoff_rows.iloc[0]['intc']
                profit_loss_amount = ((entry_price - cutoff_price) * qty)
                status = 'Profit' if cutoff_price < entry_price else 'Loss'
                results.append([stock_name, entry_time, cutoff_time, cutoff_price, f'Cut at 15:15 ({status})', profit_loss_amount])
                print(f"No SL or TGT hit. Position closed at {cutoff_time}: {cutoff_price:.2f}, PnL: {profit_loss_amount:.2f}")
            elif not stop_or_tgt_hit and cutoff_rows.empty:
                print(f"No data found for {stock_name} at the cutoff time (15:15).")

        else:
            print(f"No entry found for {stock_name} at {testing_start_time}")

    except Exception as e:
        print(f"An error occurred for {stock_name}: {e}")

# Create a DataFrame for the results
results_df = pd.DataFrame(results, columns=['Stock Name', 'Entry Time', 'Hit/Exit Time', 'Price', 'Status', 'Profit/Loss Amount'])

# Save the results to an Excel file
output_file_path = 'C:\\Users\\omkar\\Downloads\\stock_results_New_0930TO11_spec.xlsx'
results_df.to_excel(output_file_path, index=False)

print(f"Results saved to {output_file_path}")
print(f"Total results: {len(results)}")

# ==================== RESULTS SUMMARY ====================
print("\n" + "=" * 80)
print("📊 BACKTEST RESULTS SUMMARY")
print("=" * 80)
print()

total_profit_loss = results_df['Profit/Loss Amount'].sum()
winning_trades = len(results_df[results_df['Profit/Loss Amount'] > 0])
losing_trades = len(results_df[results_df['Profit/Loss Amount'] < 0])
total_trades = len(results_df)
win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0

print(f"💰 TOTAL PROFIT/LOSS: Rs. {total_profit_loss:,.2f}")
print(f"✅ WINNING TRADES: {winning_trades}")
print(f"❌ LOSING TRADES: {losing_trades}")
print(f"📈 TOTAL TRADES: {total_trades}")
print(f"🎯 WIN RATE: {win_rate:.1f}%")
print()
print("=" * 80)
print(f"📁 Files saved to: {output_file_path}")
print("=" * 80)

print(f"\nResults saved to {output_file_path}")
